# STRUM 채보 모델 실제 곡 학습 — GPU 서버 실행

STRUM의 8종 온셋 CRNN을 실제 발매곡 457곡으로 이어서 학습하고, 검증·시험 곡 채보 결과를 묶어 낸다.

- 학습은 두 개이고 동시에 돈다. `strum_p3_R`은 실제 곡 457곡만, `strum_p3_RE`는 실제 곡 + 서버에 있는 E-GMD 어쿠스틱 킷(구간의 30%)
- 결과: `~/ssaplash/results/ext_ft_results.zip` (학습 기록·로그·검증과 시험 곡 채보 결과). 가중치는 `~/ssaplash/ext_ft/runs/`에 남는다

**준비물.** JupyterHub `ssaplash/upload/`에 `ext_ft_data_01.zip` ~ `ext_ft_data_08.zip`과 `ext_ft_models.zip`을 올리고, 이 파일을 `ssaplash/notebooks/`에 올린다. 서버 `~/ssaplash/S15P31A307`은 운영 워커 코드라 덮어쓰지 않고 그 `.venv`를 실행만 한다.

**실행.** 1번 셀을 한 번 실행하면 파일 확인 → 풀기 → E-GMD 목록 → 학습 2개 → 채보 → 결과 묶기가 백그라운드로 끝까지 돈다. 브라우저를 닫아도 계속 돈다. 진행은 2번 셀로 아무 때나 본다.

**끊기면.** 서버가 다시 시작됐거나 2번 셀에 '멈춤'이 나오면 1번 셀을 다시 실행한다. 끝난 단계는 건너뛰고 학습은 마지막 저장에서 이어 간다. 이미 돌고 있을 때 1번을 다시 치면 새로 시작하지 않는다.


In [ ]:
# 1. 시작 (여러 번 실행해도 됨). 처음이면 전체 과정을 백그라운드로 시작함
#    이미 돌고 있으면 새로 시작하지 않고, 멈춰 있으면 끝난 단계는 건너뛰고 이어 감
import os
import subprocess
import sys
from pathlib import Path

GPU_ID = "6"                                # 지정받은 GPU
RUNS = ""  # 비우면 학습 2개 전부. 하나만 돌릴 때는 "strum_p3_RE"처럼
HOME = Path.home()
WORK = HOME / "ssaplash"
AI = WORK / "S15P31A307" / "AI"             # 운영 워커 코드. .venv를 실행만 함
VPY = AI / ".venv" / "bin" / "python"
EXT = WORK / "ext_ft"  # 이 작업 폴더 (풀어 둔 데이터·학습·결과)
FILES = EXT / "files"

# 실행할 스크립트와 올린 zip의 md5. 저장소 AI/scripts/train_ext의 같은 이름 파일과 내용이 같음
FILES_TEXT = {
    "_common.py": r'''
"""STRUM CRNN 학습 스크립트가 쓰는 데이터 읽기·저장 (11/11).

별도 가상환경에서 `python -I`로 돌므로 numpy·soundfile·torch만 씀. 곡 목록은 `export_manifest.py`가 만든 json.
- SegmentSet: 곡에서 임의 구간을 잘라 (파형, 프레임별 정답)을 냄. 반주를 작게 섞기·음량 바꾸기 포함
- save_acts / load_acts: 곡 하나의 프레임별 점수 저장 형식. acts_to_notes.py가 읽어 노트로 바꿈
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np

PADS = ("KICK", "SNARE", "HIHAT", "SMALL_TOM", "MID_TOM", "FLOOR_TOM", "CRASH", "RIDE")


def load_meta(path: str | Path) -> dict:
    """곡 목록 json. 음원·정답 경로가 상대 경로면 목록 파일이 있는 폴더 기준으로 바꿈(서버에 올린 묶음)."""
    path = Path(path).resolve()
    meta = json.loads(path.read_text(encoding="utf-8"))
    for song in meta["songs"]:
        for key in ("drums", "accomp", "labels"):
            if song.get(key) and not Path(song[key]).is_absolute():
                song[key] = str(path.parent / song[key])
    return meta


def load_manifest(path: str | Path) -> list[dict]:
    return load_meta(path)["songs"]


def read_mono(path: str, start: int = 0, frames: int = -1, sample_rate: int = 44100) -> np.ndarray:
    """음원 일부를 모노 float32로 읽음. start·frames는 sample_rate 기준 샘플 수.

    파일의 표본화 주파수가 다르면 같은 시간 구간만 읽어 sample_rate로 바꿈.
    """
    import soundfile as sf

    with sf.SoundFile(path) as f:
        sr = f.samplerate
        if sr == sample_rate:
            f.seek(min(start, f.frames))
            return f.read(frames, dtype="float32", always_2d=True).mean(axis=1)
        f.seek(min(int(round(start * sr / sample_rate)), f.frames))
        n = -1 if frames < 0 else int(np.ceil(frames * sr / sample_rate))
        audio = f.read(n, dtype="float32", always_2d=True).mean(axis=1)
    from math import gcd

    from scipy.signal import resample_poly

    g = gcd(sample_rate, sr)
    out = resample_poly(audio, sample_rate // g, sr // g).astype(np.float32)
    return out if frames < 0 else out[:frames]


def frame_targets(
    times: np.ndarray,
    pads: np.ndarray,
    n_frames: int,
    fps: float,
    n_classes: int,
    pad_to_class: dict[int, int],
    radiation: int = 0,
    floor: bool = False,
) -> np.ndarray:
    """노트 → (프레임, 분류) 0/1. radiation만큼 앞뒤 프레임도 1."""
    target = np.zeros((n_frames, n_classes), dtype=np.float32)
    pos = times * fps
    frames = np.floor(pos).astype(int) if floor else np.rint(pos).astype(int)
    for f, p in zip(frames.tolist(), pads.tolist(), strict=True):
        c = pad_to_class.get(int(p))
        if c is None:
            continue
        lo, hi = max(0, f - radiation), min(n_frames, f + radiation + 1)
        if lo < hi:
            target[lo:hi, c] = 1.0
    return target


class SegmentSet:
    """곡 목록에서 임의 구간을 뽑는 데이터. torch Dataset처럼 쓰되 색인은 무시하고 매번 새로 뽑음."""

    def __init__(
        self,
        manifests: list[tuple[str, float]],
        seconds: float,
        sample_rate: int = 44100,
        accomp_prob: float = 0.3,
        accomp_gain: tuple[float, float] = (0.05, 0.3),
        gain_db: tuple[float, float] = (-6.0, 6.0),
        length: int = 1000,
        seed: int = 0,
    ):
        self.groups = []
        for path, weight in manifests:
            songs = [s for s in load_manifest(path) if s["duration"] > seconds + 1.0]
            self.groups.append((songs, float(weight)))
        self.weights = np.array([w for _, w in self.groups], dtype=np.float64)
        self.weights /= self.weights.sum()
        self.seconds, self.sr = seconds, sample_rate
        self.accomp_prob, self.accomp_gain, self.gain_db = accomp_prob, accomp_gain, gain_db
        self.length, self.seed = length, seed
        self._labels: dict[str, tuple[np.ndarray, np.ndarray]] = {}
        self._rng: np.random.Generator | None = None

    def __len__(self) -> int:
        return self.length

    def labels(self, song: dict) -> tuple[np.ndarray, np.ndarray]:
        key = song["labels"]
        if key not in self._labels:
            z = np.load(key)
            self._labels[key] = (z["time"], z["pad"])
        return self._labels[key]

    def draw(self) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """(파형, 구간 안 노트 시각(구간 시작 기준 초), 패드 번호)."""
        if self._rng is None:
            import os

            self._rng = np.random.default_rng([self.seed, os.getpid()])
        rng = self._rng
        songs, _ = self.groups[int(rng.choice(len(self.groups), p=self.weights))]
        song = songs[int(rng.integers(len(songs)))]
        n = int(self.seconds * self.sr)
        total = int(song["duration"] * self.sr)
        start = int(rng.integers(0, max(total - n, 1)))
        audio = read_mono(song["drums"], start, n, self.sr)
        if len(audio) < n:
            audio = np.pad(audio, (0, n - len(audio)))
        if song.get("accomp") and rng.random() < self.accomp_prob:
            other = read_mono(song["accomp"], start, n, self.sr)
            if len(other) < n:
                other = np.pad(other, (0, n - len(other)))
            audio = audio + float(rng.uniform(*self.accomp_gain)) * other
        audio = audio * float(10 ** (rng.uniform(*self.gain_db) / 20))
        times, pads = self.labels(song)
        t0 = start / self.sr
        keep = (times >= t0) & (times < t0 + self.seconds)
        return audio.astype(np.float32), times[keep] - t0, pads[keep]


def save_acts(
    path: Path,
    fps: float,
    names: list[str],
    acts: dict[str, np.ndarray],
    seconds: float = 0.0,
    checkpoint: str = "",
) -> None:
    """프레임별 점수 저장. acts 값은 (프레임, len(names)) 0~1. seconds는 채보에 걸린 시간."""
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        fps=float(fps),
        names=np.array(names, dtype="U16"),
        seconds=float(seconds),
        checkpoint=str(checkpoint),
        **{f"acts_{k}": v.astype(np.float16) for k, v in acts.items()},
    )


def acts_path(root: Path, tag: str, manifest: str, song_id: str) -> Path:
    return root / tag / manifest / f"{song_id.replace('/', '__')}.npz"''',
    "_loop.py": r'''
"""STRUM CRNN 학습 스크립트가 쓰는 학습·검증·점수 저장 흐름 (11/11).

`strum_crnn.py`는 (1) 모델 읽기 (2) 파형 → 분류별 온셋 로짓 (3) 곡 하나 → 프레임별 점수 세 가지만 만들고
나머지(구간 뽑기, focal BCE, 검증 곡 8패드 F1, 가장 높은 가중치 저장, 끊긴 학습 이어 가기)는 여기 것을 씀.
별도 가상환경에서 `python -I`로 돌므로 numpy·scipy·torch만 씀.
"""

from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
from _common import (
    PADS,
    SegmentSet,
    acts_path,
    frame_targets,
    load_manifest,
    load_meta,
    read_mono,
    save_acts,
)

THRESHOLDS = (0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)


class TrainSet:
    """DataLoader용. 구간 파형과 프레임 정답 (프레임, 분류)을 냄."""

    def __init__(
        self,
        segments: SegmentSet,
        hop: int,
        fps: float,
        n_classes: int,
        pad_to_class: dict[int, int],
        radiation: int,
        floor: bool,
    ):
        self.segments, self.hop, self.fps = segments, hop, fps
        self.n_classes, self.pad_to_class = n_classes, pad_to_class
        self.radiation, self.floor = radiation, floor

    def __len__(self) -> int:
        return len(self.segments)

    def __getitem__(self, _):
        import torch

        audio, times, pads = self.segments.draw()
        n_frames = len(audio) // self.hop + 1
        target = frame_targets(
            times,
            pads,
            n_frames,
            self.fps,
            self.n_classes,
            self.pad_to_class,
            self.radiation,
            self.floor,
        )
        return torch.from_numpy(audio), torch.from_numpy(target)


def focal_bce(logits, targets, pos_weight, gamma: float = 2.0):
    """분류별 pos_weight를 준 focal BCE. logits·targets는 (배치, 프레임, 분류)."""
    import torch
    import torch.nn.functional as F

    logits, targets = logits.float(), targets.float()
    bce = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight, reduction="none"
    )
    probs = torch.sigmoid(logits)
    p_t = torch.where(targets > 0.5, probs, 1 - probs).clamp(min=1e-6, max=1 - 1e-6)
    return ((1 - p_t) ** gamma * bce).mean()


def match_counts(truth: np.ndarray, pred: np.ndarray, window: float = 0.05) -> tuple[int, int, int]:
    """정렬된 두 시각 배열을 ±window 안에서 앞에서부터 짝지은 tp·fp·fn."""
    i = j = tp = 0
    while i < len(truth) and j < len(pred):
        d = pred[j] - truth[i]
        if abs(d) <= window:
            tp, i, j = tp + 1, i + 1, j + 1
        elif d < 0:
            j += 1
        else:
            i += 1
    return tp, len(pred) - tp, len(truth) - tp


def evaluate(infer, songs: list[dict], names: list[str], fps: float) -> dict:
    """검증 곡 8패드 F1. infer(파형) → (프레임, 분류) 점수. 분류별 문턱값은 이 곡들에서 가장 높은 것(가중치 고르기 전용)."""
    from scipy.signal import find_peaks

    gap = max(1, int(round(0.03 * fps)))
    counts = np.zeros((len(names), len(THRESHOLDS), 3), dtype=np.int64)
    for song in songs:
        acts = infer(read_mono(song["drums"]))
        z = np.load(song["labels"])
        for c, cname in enumerate(names):
            truth = np.sort(z["time"][z["pad"] == PADS.index(cname)])
            peaks, props = find_peaks(acts[:, c], height=THRESHOLDS[0], distance=gap)
            for ti, th in enumerate(THRESHOLDS):
                pred = peaks[props["peak_heights"] >= th] / fps
                counts[c, ti] += match_counts(truth, pred)
    out, total = {}, np.zeros(3, dtype=np.int64)
    for c, cname in enumerate(names):
        f1 = 2 * counts[c, :, 0] / np.maximum(2 * counts[c, :, 0] + counts[c, :, 1:].sum(1), 1)
        best = int(np.argmax(f1))
        total += counts[c, best]
        out[cname] = {"f1": float(f1[best]), "thr": THRESHOLDS[best]}
    out["f1"] = float(2 * total[0] / max(2 * total[0] + total[1] + total[2], 1))
    return out


def dump(
    infer_all,
    manifests: list[str],
    acts_root: str,
    tag: str,
    names: list[str],
    fps: float,
    checkpoint: str,
) -> None:
    """곡 목록마다 프레임별 점수를 저장. infer_all(파형) → {종류: (프레임, 분류)}. 이미 있는 곡은 건너뜀."""
    import torch

    for manifest in manifests:
        meta = load_meta(manifest)
        done = 0
        for song in meta["songs"]:
            path = acts_path(Path(acts_root), tag, meta["name"], song["song_id"])
            if path.is_file():
                continue
            audio = read_mono(song["drums"])
            t0 = time.perf_counter()
            acts = infer_all(audio)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            save_acts(path, fps, names, acts, time.perf_counter() - t0, checkpoint)
            done += 1
        print(f"[dump] {tag} {meta['name']} 새로 {done}곡", flush=True)


def add_common_args(sub, default_seconds: float, default_lr: float, default_batch: int) -> None:
    """dump·train 하위 명령에 공통 인자를 붙임."""
    d, t = sub.choices["dump"], sub.choices["train"]
    for p in (d, t):
        p.add_argument("--device", default="cuda")
    t.add_argument(
        "--mem-fraction",
        type=float,
        default=0.0,
        help="이 프로세스가 쓸 GPU 메모리 상한(전체 대비). 0이면 상한 없음. 로컬 8GB GPU는 0.85",
    )
    d.add_argument("--tag", required=True)
    d.add_argument("--manifest", action="append", required=True)
    d.add_argument("--checkpoint", default=None, help="이어서 학습한 가중치(best.pt)")
    d.add_argument("--acts", default="data/ext_train/acts")
    t.add_argument("--train", action="append", required=True, help="곡 목록[:가중치] (여러 번)")
    t.add_argument("--val", required=True)
    t.add_argument("--out", required=True)
    t.add_argument("--steps", type=int, default=6000)
    t.add_argument("--batch", type=int, default=default_batch)
    t.add_argument("--seconds", type=float, default=default_seconds)
    t.add_argument("--lr", type=float, default=default_lr)
    t.add_argument("--radiation", type=int, default=1)
    t.add_argument("--pos-scale", type=float, default=1.0)
    t.add_argument("--gamma", type=float, default=2.0)
    t.add_argument("--accomp-prob", type=float, default=0.3)
    t.add_argument("--workers", type=int, default=4)
    t.add_argument("--eval-every", type=int, default=500)
    t.add_argument("--log-every", type=int, default=50)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument(
        "--amp", action="store_true", help="모델 계산을 16비트로(메모리 절약). 손실은 32비트"
    )


def resolve_paths(args: argparse.Namespace) -> None:
    """경로를 절대 경로로 바꿈. 학습 목록 `경로:비율`은 (경로, 비율)로."""
    args.root = str(Path.cwd())
    for key in ("checkpoint", "acts", "val", "out", "init"):
        if getattr(args, key, None):
            setattr(args, key, str(Path(getattr(args, key)).resolve()))
    if getattr(args, "manifest", None):
        args.manifest = [str(Path(m).resolve()) for m in args.manifest]
    if getattr(args, "train", None):
        fixed = []
        for item in args.train:
            path, sep, weight = item.rpartition(":")
            if sep and weight.replace(".", "", 1).isdigit():
                fixed.append((str(Path(path).resolve()), float(weight)))
            else:
                fixed.append((str(Path(item).resolve()), 1.0))
        args.train = fixed


def train(
    args: argparse.Namespace,
    model,
    logits_of,
    infer,
    names: list[str],
    hop: int,
    fps: float,
    pad_to_class: dict[int, int],
    pos_weight: list[float],
    floor: bool = False,
) -> int:
    """학습 흐름. logits_of(파형 배치) → (배치, 프레임, 분류) 로짓, infer(파형) → (프레임, 분류) 점수."""
    import torch
    from torch.utils.data import DataLoader

    device = torch.device(args.device)
    if device.type == "cuda" and getattr(args, "mem_fraction", 0):
        # Windows에서 GPU 메모리가 넘치면 시스템 메모리로 새서 PC가 멈출 수 있어, 넘으면 이 프로세스만 오류로 끝나게 함
        torch.cuda.set_per_process_memory_fraction(args.mem_fraction)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    segments = SegmentSet(
        args.train,
        args.seconds,
        accomp_prob=args.accomp_prob,
        length=args.steps * args.batch,
        seed=args.seed,
    )
    loader = DataLoader(
        TrainSet(segments, hop, fps, len(names), pad_to_class, args.radiation, floor),
        batch_size=args.batch,
        num_workers=args.workers,
        persistent_workers=args.workers > 0,
        drop_last=True,
    )
    val_songs = load_manifest(args.val)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=args.lr,
        total_steps=args.steps,
        pct_start=max(0.05, 2.0 / args.steps),
        anneal_strategy="cos",
    )
    weight = torch.tensor(pos_weight, dtype=torch.float32, device=device) * args.pos_scale
    log = out / "train_log.jsonl"
    step, best = 0, -1.0
    last = out / "last.pt"

    def run_eval() -> dict:
        model.eval()
        with torch.no_grad():
            val = evaluate(infer, val_songs, names, fps)
        model.train()
        return val

    if last.is_file():
        state = torch.load(last, map_location=device)
        model.load_state_dict(state["model_state_dict"])
        opt.load_state_dict(state["optimizer"])
        sched.load_state_dict(state["scheduler"])
        step, best = int(state["step"]), float(state["best"])
        print(f"[train] {last}에서 이어 감 (스텝 {step}, 최고 {best:.4f})", flush=True)
    else:
        saved = {
            k: v for k, v in vars(args).items() if isinstance(v, (str, int, float, bool, list))
        }
        (out / "args.json").write_text(json.dumps(saved, indent=1), encoding="utf-8")
        base = run_eval()
        best = base["f1"]
        torch.save({"model_state_dict": model.state_dict(), "step": 0}, out / "best.pt")
        with open(log, "a", encoding="utf-8") as fh:
            fh.write(json.dumps({"step": 0, "val": base}) + "\n")
        per = " ".join(f"{k[:4]} {v['f1']:.3f}" for k, v in base.items() if k != "f1")
        print(f"[train] 학습 전 검증 8패드 F1 {best:.4f} | {per}", flush=True)

    model.train()
    scaler = torch.amp.GradScaler(device.type, enabled=args.amp)
    running, t0 = 0.0, time.perf_counter()
    for audio, target in loader:
        if step >= args.steps:
            break
        audio, target = audio.to(device), target.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=args.amp):
            logits = logits_of(audio)
            n = min(logits.shape[1], target.shape[1])
            loss = focal_bce(logits[:, :n], target[:, :n], weight, args.gamma)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        step += 1
        running += float(loss.detach())
        if step % args.log_every == 0:
            rate = (time.perf_counter() - t0) / args.log_every
            print(
                f"[train] 스텝 {step}/{args.steps} 손실 {running / args.log_every:.5f} "
                f"스텝당 {rate:.2f}초 lr {sched.get_last_lr()[0]:.2e}",
                flush=True,
            )
            running, t0 = 0.0, time.perf_counter()
        if step % args.eval_every == 0 or step == args.steps:
            val = run_eval()
            with open(log, "a", encoding="utf-8") as fh:
                fh.write(json.dumps({"step": step, "val": val}) + "\n")
            mark = ""
            if val["f1"] > best:
                best = val["f1"]
                torch.save({"model_state_dict": model.state_dict(), "step": step}, out / "best.pt")
                mark = " (최고)"
            per = " ".join(f"{k[:4]} {v['f1']:.3f}" for k, v in val.items() if k != "f1")
            print(f"[train] 스텝 {step} 검증 8패드 F1 {val['f1']:.4f}{mark} | {per}", flush=True)
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer": opt.state_dict(),
                    "scheduler": sched.state_dict(),
                    "step": step,
                    "best": best,
                },
                last,
            )
            t0 = time.perf_counter()
    print(f"[train] 끝. 최고 검증 8패드 F1 {best:.4f} → {out / 'best.pt'}", flush=True)
    # 끝났다는 표시. 데이터 읽기 작업을 정리하다 프로세스가 비정상 종료해도(Windows) 학습은 끝난 것으로 봄
    (out / "done.json").write_text(json.dumps({"step": step, "best": best}), encoding="utf-8")
    return 0''',
    "strum_crnn.py": r'''
"""STRUM의 8종 온셋 CRNN 한 개만 써서 채보하고 실제 곡으로 이어서 학습 (별도 환경 `.venv-strum`, 11/11).

STRUM 전체 파이프라인(온셋 검출 → 분류기 6개 → 드럼 킷 분리 아비터 → 후처리)은 곡 길이의 0.2 ~ 0.5배가 걸리고
이어서 학습하기 어려움. 저장소의 `checkpoints/drums_phase3/best.pt`는 8종 온셋을 프레임별로 바로 내는
CRNN임(42M, MIT). 출력 mc_onset_probs

dump: 곡 목록의 곡마다 프레임별 8종 온셋 점수를 `data/ext_train/acts/<태그>/<목록>/<곡>.npz`로 저장. 점수 종류 이름은 p3
train: 임의 구간(--seconds)으로 8종 온셋 머리까지 전체를 이어서 학습. 손실은 저장소와 같은 focal BCE.
  --eval-every 스텝마다 검증 곡을 채보해 8패드 F1(분류별 문턱값은 검증 곡에서 고름)을 찍고 가장 높은 가중치를 best.pt로 둠

  .venv-strum/Scripts/python.exe -I scripts/train_ext/strum_crnn.py dump --tag strum_p3 --manifest data/ext_train/val10.json
  .venv-strum/Scripts/python.exe -I scripts/train_ext/strum_crnn.py train --train data/ext_train/train457.json --val data/ext_train/val10.json --out data/models/ext_ft/strum_p3_real
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))

import _loop  # noqa: E402
import numpy as np  # noqa: E402
from _common import PADS  # noqa: E402

SR, N_FFT, HOP, N_MELS = 44100, 2048, 512, 128
FPS = SR / HOP
# STRUM 분류 순서(Kick, Snare, HiHat, HighTom, Ride, LowTom, Crash, FloorTom) → 우리 패드 이름
NAMES = ["KICK", "SNARE", "HIHAT", "SMALL_TOM", "RIDE", "MID_TOM", "CRASH", "FLOOR_TOM"]
PAD_TO_CLASS = {PADS.index(n): i for i, n in enumerate(NAMES)}
POS_WEIGHT = [3, 4, 5, 10, 8, 10, 8, 10]  # phase3 체크포인트 config의 onset_class_pos_weights


WEIGHTS = "data/models/strum/drums/drums_phase3/best.pt"


def load_module(model_py: Path):
    """저장소 `src/models/drums_v13.py` 한 파일만 불러옴(torch만 필요). 저장소의 다른 코드는 안 씀."""
    import importlib.util

    spec = importlib.util.spec_from_file_location("strum_drums_v13", model_py)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def load_model(module, weights: Path, device, checkpoint: str | None = None):
    """저장소 batch_infer_hybrid.py의 load_phase3_model과 같은 방식으로 모델을 만듦."""
    import torch

    # 공개 가중치 파일에는 텐서와 numpy 스칼라·dtype만 들어 있음. 그 밖의 객체가 든 파일은 읽지 않음
    core = getattr(np, "_core", None) or np.core  # numpy 1.x에는 _core가 없음
    safe = [np.dtype, core.multiarray.scalar]
    safe += [type(np.dtype(t)) for t in ("float64", "float32", "int64", "int32", "bool")]
    torch.serialization.add_safe_globals(safe)
    ckpt = torch.load(weights, map_location="cpu", weights_only=True)
    cfg = (ckpt.get("config") or {}).get("model", {})
    model = module.TwoStageDrumsCRNN(
        n_mels=cfg.get("n_mels", 128),
        conv_channels=cfg.get("conv_channels", [64, 128, 256, 512]),
        freq_subbands=cfg.get("freq_subbands", [32, 64, 96, 128]),
        subband_proj_dim=cfg.get("subband_proj_dim", 256),
        lstm_hidden=cfg.get("lstm_hidden", 640),
        lstm_layers=cfg.get("lstm_layers", 3),
        attention_heads=cfg.get("attention_heads", 10),
        attention_type=cfg.get("attention_type", "flash"),
        attention_window=cfg.get("attention_window", 512),
        dropout=0.0,
        onset_detector_hidden=cfg.get("onset_detector_hidden", 320),
        classifier_hidden=cfg.get("classifier_hidden", 640),
        num_classes=cfg.get("num_classes", 8),
        predict_velocity=cfg.get("predict_velocity", False),
    )
    model.add_multiclass_onset_head()
    missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
    if missing or unexpected:
        raise RuntimeError(f"STRUM 가중치와 층 이름이 안 맞음: 없음 {missing} 남음 {unexpected}")
    name = "opria123/strum phase3"
    if checkpoint:
        state = torch.load(checkpoint, map_location="cpu", weights_only=True)
        model.load_state_dict(state["model_state_dict"], strict=True)
        name = str(checkpoint)
    return model.to(device), name


class Mel:
    def __init__(self, device):
        import torchaudio

        self.t = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS
        ).to(device)

    def __call__(self, audio):
        """(배치, 샘플) → (배치, 1, 멜, 프레임) 로그 멜. 저장소 bg_mel(배경 빼기 꺼짐)과 같은 계산."""
        import torch

        return torch.log(self.t(audio) + 1e-8).unsqueeze(1)


def infer(model, mel: Mel, audio: np.ndarray, device, batch: int = 8) -> dict:
    """곡 하나 → 프레임별 8종 온셋 점수 {"p3": (프레임, 8)}. 저장소 추론과 같이 10초 구간을 절반씩 겹쳐 평균."""
    import torch

    with torch.no_grad():
        spec = mel(torch.from_numpy(audio).to(device)[None])[0]  # (1, 멜, 프레임)
        total = spec.shape[-1]
        seg = int(10.0 * SR / HOP)
        hop = seg // 2
        starts = list(range(0, max(total - seg, 0) + 1, hop)) or [0]
        if starts[-1] + seg < total:
            starts.append(starts[-1] + hop)
        mc = torch.zeros(total, 8, device=device)
        count = torch.zeros(total, 1, device=device)
        for i in range(0, len(starts), batch):
            chunk = starts[i : i + batch]
            parts = [spec[:, :, s : s + seg] for s in chunk]
            x = torch.stack([torch.nn.functional.pad(p, (0, seg - p.shape[-1])) for p in parts])
            out = model(x)
            for k, s in enumerate(chunk):
                n = min(seg, total - s)
                mc[s : s + n] += out["mc_onset_probs"][k, :n]
                count[s : s + n] += 1
        count = count.clamp(min=1)
    return {"p3": (mc / count).cpu().numpy()}


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    sub = ap.add_subparsers(dest="cmd", required=True)
    for cmd in ("dump", "train"):
        p = sub.add_parser(cmd)
        p.add_argument("--model-py", default="ext/strum/src/models/drums_v13.py")
        p.add_argument("--weights", default=WEIGHTS, help="공개 가중치 파일")
    _loop.add_common_args(sub, default_seconds=20.0, default_lr=3e-5, default_batch=4)
    t = sub.choices["train"]
    t.add_argument("--dropout", type=float, default=0.3)
    t.add_argument("--spec-augment", action="store_true")
    t.add_argument("--init", default=None, help="이 가중치에서 시작(다른 학습의 best.pt)")
    args = ap.parse_args()
    _loop.resolve_paths(args)

    import torch

    module = load_module(Path(args.model_py).resolve())
    device = torch.device(args.device if torch.cuda.is_available() else "cpu")
    start = args.checkpoint if args.cmd == "dump" else args.init
    model, name = load_model(module, Path(args.weights).resolve(), device, start)
    mel = Mel(device)

    if args.cmd == "dump":
        model.eval()
        _loop.dump(
            lambda audio: infer(model, mel, audio, device),
            args.manifest,
            args.acts,
            args.tag,
            NAMES,
            FPS,
            name,
        )
        return 0

    for m in model.modules():
        if isinstance(m, torch.nn.Dropout):
            m.p = args.dropout
    model.lstm.dropout = args.dropout
    freq_mask = time_mask = None
    if args.spec_augment:
        import torchaudio

        freq_mask = torchaudio.transforms.FrequencyMasking(27)
        time_mask = torchaudio.transforms.TimeMasking(50)

    def logits_of(audio):
        with torch.no_grad():
            x = mel(audio)
            if freq_mask is not None:
                low = float(x.min())
                x = time_mask(freq_mask(x, mask_value=low), mask_value=low)
        return model(x)

    return _loop.train(
        args,
        model,
        lambda audio: logits_of(audio)["mc_onset_logits"],
        lambda audio: infer(model, mel, audio, device)["p3"],
        NAMES,
        HOP,
        FPS,
        PAD_TO_CLASS,
        POS_WEIGHT,
        floor=True,
    )


if __name__ == "__main__":
    raise SystemExit(main())''',
    "export_manifest.py": r'''
"""STRUM CRNN 학습·채보용 곡 목록(manifest) 만들기 (11/11).

STRUM은 가상환경이 달라 `ssaplash_ai`를 못 부르므로, 곡마다 드럼 음원 경로·반주 경로·정답 노트를
파일로 풀어 둠. 학습 스크립트(`scripts/train_ext/strum_crnn.py`)는 이 json만 읽음.

- Songsterr 계열(kind: songsterr)은 곡 폴더의 `drums.flac`(서비스 분리 드럼)·`drumless.flac` 경로를 그대로 적음
- 그 밖(MDB·ENST 등)은 저장 특징의 input 트랙(서비스와 같은 입력)을 flac으로 써서 그 경로를 적음
- `--audio original`이면 데이터셋의 드럼 음원 파일 경로를 그대로 적음(E-GMD 학습 분할처럼 곡이 많을 때)
- 정답은 `<out>/labels/<데이터셋>/<곡>.npz`: time(초), pad(0~7, 아래 PADS 순서), velocity. 페달 하이햇은 하이햇으로 합침

  uv run python scripts/train_ext/export_manifest.py --dataset configs/bench/datasets/public_songs_train_full.yaml --name train457
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

PADS = ("KICK", "SNARE", "HIHAT", "SMALL_TOM", "MID_TOM", "FLOOR_TOM", "CRASH", "RIDE")


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--dataset", required=True)
    ap.add_argument("--name", required=True, help="목록 이름. <out>/<name>.json으로 저장")
    ap.add_argument("--features", default="data/rule_features", help="Songsterr가 아닌 곡의 입력")
    ap.add_argument("--out", type=Path, default=Path("data/ext_train"))
    ap.add_argument("--enst-c1-ride", action="store_true")
    ap.add_argument(
        "--audio",
        choices=("features", "original"),
        default="features",
        help="Songsterr가 아닌 곡의 음원. original은 데이터셋 드럼 파일 경로(required_files 첫 번째)",
    )
    args = ap.parse_args()

    import soundfile as sf

    if args.enst_c1_ride:
        import ssaplash_ai.bench.datasets.enst as enst

        enst.ENST_LABEL_PAD["c1"] = "RIDE"
        enst.ENST_LABEL_PAD["c4"] = "RIDE"
    # 서버 운영 코드가 오래된 판이어도 돌도록, 쓰는 모듈만 그때그때 불러옴
    from ssaplash_ai.bench.datasets import DatasetConfig, load_dataset

    cfg = DatasetConfig.from_yaml(args.dataset)
    ds = load_dataset(cfg)
    kind = str(cfg.kind)
    index = {p: i for i, p in enumerate(PADS)}
    songs = []
    for sid in ds.song_ids():
        safe = sid.replace("/", "__")
        if args.audio == "original" and kind != "songsterr":
            # 드럼 음원이나 정답 파일(앞의 두 개)이 없는 곡은 뺌. 반주는 쓰지 않으므로 없어도 됨
            if not all(Path(f).is_file() for f in ds.required_files(sid)[:2]):
                continue
        hits = ds.hits_of(sid)[0].copy()
        hits["pad"][hits["pad"] == "HIHAT_PEDAL"] = (
            "HIHAT"  # 8패드 채점과 같이 페달을 하이햇에 합침
        )
        keep = np.isin(hits["pad"], PADS)
        hits = hits[keep]
        label = args.out / "labels" / args.name / f"{safe}.npz"
        label.parent.mkdir(parents=True, exist_ok=True)
        np.savez(
            label,
            time=hits["time_sec"].astype(np.float64),
            pad=np.array([index[p] for p in hits["pad"]], dtype=np.int16),
            velocity=hits["velocity"].astype(np.int16),
        )
        accomp = None
        if kind == "songsterr":
            from ssaplash_ai.bench.datasets.songsterr import final_dir

            f = final_dir(Path(cfg.root) / sid)
            drums, accomp = f / "drums.flac", f / "drumless.flac"
            duration = sf.info(str(drums)).duration
        elif args.audio == "original":
            drums = ds.required_files(sid)[0]
            try:
                info = sf.info(str(drums))
            except Exception:  # 못 여는 파일은 뺌
                continue
            # 표본화 주파수가 달라도 넣음. 읽을 때 _common.read_mono가 바꿈
            duration = info.duration
        else:
            drums = args.out / "audio" / args.name / f"{safe}.flac"
            if not drums.is_file():
                from ssaplash_ai.bench.analysis.rule_features import feature_path, load_features

                feats = load_features(feature_path(Path(args.features), ds.name, sid))
                if "input" in feats.stems:
                    audio = np.atleast_2d(feats.stems["input"].astype(np.float32))
                else:
                    audio = np.atleast_2d(ds.load(sid).drums_true.astype(np.float32))
                drums.parent.mkdir(parents=True, exist_ok=True)
                sf.write(drums, audio.mean(axis=0), feats.sample_rate, subtype="PCM_16")
            duration = sf.info(str(drums)).duration
        songs.append(
            {
                "song_id": sid,
                "drums": str(Path(drums).resolve()),
                "accomp": str(Path(accomp).resolve()) if accomp else None,
                "labels": str(label.resolve()),
                "duration": float(duration),
                "n_hits": int(len(hits)),
            }
        )
    out = args.out / f"{args.name}.json"
    out.write_text(
        json.dumps({"name": args.name, "dataset": ds.name, "pads": PADS, "songs": songs}, indent=1),
        encoding="utf-8",
    )
    hours = sum(s["duration"] for s in songs) / 3600
    print(f"[manifest] {args.name} 곡 {len(songs)} {hours:.1f}시간 → {out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())''',
    "ext_ft_pipeline.py": r'''
"""STRUM CRNN 실제 곡 학습 전체 과정을 한 번에 돌림 (11/11).

단계를 차례로 돌고, 끝난 단계마다 {work}/done/{단계}.json을 남김. 다시 실행하면 끝난 단계는 건너뛰고
학습은 각자 마지막 저장(last.pt)에서 이어 감. 운영 코드 폴더(--ai)는 .venv와 ssaplash_ai를 실행만 하고 아무것도 쓰지 않음.

단계
- check: GPU와 필요한 패키지 확인. 없는 패키지는 {work}/pylib에 받아 봄
- data: 올린 zip의 md5를 확인하고 {work}/ext_ft_data(음원·정답)·{work}/ext_ft_models(가중치·모델 코드)로 풂
- egmd: 서버에 있는 E-GMD 어쿠스틱 킷 학습 분할의 곡 목록·정답 만들기. 실패하면 E-GMD를 섞는 학습만 뺌
- train: 학습을 전부 동시에 실행. strum_p3_R은 실제 곡 457곡만, strum_p3_RE는 실제 곡 + E-GMD(구간의 30%)
- dump: 학습마다 검증 점수가 가장 높은 가중치로 검증·시험 곡을 채보해 프레임별 점수 저장
- pack: 학습 기록·로그·채보 결과를 {results}/{zip-name}.zip으로 묶음 (가중치는 {work}/runs에 남김)

  python ext_ft_pipeline.py run --work ~/ssaplash/ext_ft --upload ~/ssaplash/upload --ai ~/ssaplash/S15P31A307/AI \
      --egmd-root ~/ssaplash/datasets/e-gmd/e-gmd-v1.0.0 --results ~/ssaplash/results
  python ext_ft_pipeline.py status --work ~/ssaplash/ext_ft
"""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import shlex
import subprocess
import time
import zipfile
from pathlib import Path

HERE = Path(__file__).resolve().parent
EVAL_SETS = ("val10", "test31", "holdout38", "mdb", "enst", "enst_sep")
# 불러와야 하는 패키지 (import 이름 → 받을 때 쓰는 이름)
NEEDS = {
    "strum": {
        "torch": "torch",
        "torchaudio": "torchaudio",
        "scipy": "scipy",
        "soundfile": "soundfile",
    },
}
# 운영 환경과 버전이 맞아야 해서 받지 않는 패키지. 없으면 학습을 시작하지 않음
HEAVY = {"torch", "torchaudio", "numpy", "scipy"}
MAX_FETCH = 25  # 한 모델에서 차례로 받아 보는 패키지 수 한도


def run_specs(args: argparse.Namespace) -> dict[str, dict]:
    """학습 이름 → 모델 종류·스크립트·인자. 스텝 수는 --scale로 줄일 수 있음(동작 확인용)."""
    models = args.work / "ext_ft_models"

    def steps(n: int) -> str:
        return str(max(int(n * args.scale), 20))

    def every(n: int) -> str:
        return str(max(int(n * args.scale), 10))

    small = args.scale < 0.05  # 동작 확인용: 배치도 줄임
    strum = ["--model-py", str(models / "strum" / "drums_v13.py"), "--batch", "2" if small else "8", "--seconds", "10",
             "--steps", steps(8000), "--eval-every", every(500), "--lr", "3e-5", "--spec-augment",
             "--accomp-prob", "0"]  # fmt: skip
    p3 = ["--weights", str(models / "strum" / "phase3.pt")]
    out = {}
    # _R은 실제 곡만, _RE는 실제 곡 + E-GMD. 서비스 가중치는 strum_p3_RE의 best.pt
    for suffix in ("R", "RE"):
        out[f"strum_p3_{suffix}"] = {
            "model": "strum",
            "script": "strum_crnn.py",
            "train": [*p3, *strum],
            "dump": [*p3, "--model-py", strum[1]],
            "egmd": suffix == "RE",
            "share": 0.3,
        }
    return out


def log(text: str) -> None:
    print(f"[{time.strftime('%m-%d %H:%M:%S')}] {text}", flush=True)


class Ctx:
    def __init__(self, args: argparse.Namespace):
        self.args = args
        self.work: Path = args.work
        self.done = self.work / "done"
        self.runs = self.work / "runs"
        self.data = self.work / "ext_ft_data"
        self.state_path = self.work / "state.json"
        for d in (self.done, self.runs, self.work / "logs"):
            d.mkdir(parents=True, exist_ok=True)

    def state(self) -> dict:
        if self.state_path.is_file():
            return json.loads(self.state_path.read_text(encoding="utf-8"))
        return {}

    def save_state(self, **kw) -> None:
        st = self.state()
        st.update(kw)
        self.state_path.write_text(json.dumps(st, ensure_ascii=False, indent=1), encoding="utf-8")

    def is_done(self, stage: str) -> bool:
        return (self.done / f"{stage}.json").is_file()

    def mark_done(self, stage: str, info: dict) -> None:
        (self.done / f"{stage}.json").write_text(
            json.dumps(info, ensure_ascii=False, indent=1), encoding="utf-8"
        )

    def env(self) -> dict:
        env = dict(os.environ, PYTHONUTF8="1", PYTHONUNBUFFERED="1")
        for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
            env[key] = "1"  # 데이터 읽기 작업마다 코어를 다 쓰지 않게 (공용 서버)
        pylib = self.work / "pylib"
        env["PYTHONPATH"] = str(pylib) if pylib.is_dir() else ""
        return env

    def python(self) -> list[str]:
        return shlex.split(self.args.vpy)


# ---------- 단계 ----------


def first_problem(ctx: Ctx, statements: list[str]) -> tuple[str, str, str] | None:
    """운영 python에서 처음 실패하는 문장. (문장, 없는 모듈 이름, 오류 글). 전부 되면 None.

    안쪽 오류를 감싸 다시 던지는 패키지가 있어, 원인 사슬을 따라가 가장 안쪽의 없는 모듈을 찾음.
    """
    lines = [
        "import json, re",
        "def inner(e):",
        "    name, n = '', 0",
        "    while e is not None and n < 12:",
        "        if isinstance(e, ModuleNotFoundError) and e.name:",
        "            name = e.name",
        "        else:",
        "            m = re.search(r\"No module named '([^']+)'\", str(e))",
        "            name = m.group(1) if m else name",
        "        e, n = (e.__cause__ or e.__context__), n + 1",
        "    return name",
        "found = None",
        f"for s in {statements!r}:",
        "    try:",
        "        exec(s, {})",
        "    except Exception as e:",
        "        found = [s, inner(e), (type(e).__name__ + ': ' + str(e))[-400:]]",
        "        break",
        "print('PROBLEM=' + json.dumps(found))",
    ]
    code = chr(10).join(lines)
    out = subprocess.run([*ctx.python(), "-c", code], capture_output=True, text=True, env=ctx.env())
    for line in out.stdout.splitlines():
        if line.startswith("PROBLEM="):
            found = json.loads(line[len("PROBLEM=") :])
            return tuple(found) if found else None
    return (statements[0], "", out.stderr.strip()[-400:])


def fetch(ctx: Ctx, package: str) -> bool:
    """패키지 하나를 {work}/pylib에 받음(의존 패키지는 따로 하나씩 확인해 받음)."""
    pylib = ctx.work / "pylib"
    cmd = [*shlex.split(ctx.args.uv), "pip", "install", "--python", ctx.python()[0],
           "--target", str(pylib), "--no-deps", package]  # fmt: skip
    got = subprocess.run(cmd, capture_output=True, text=True)
    if got.returncode != 0:
        log(f"  {package} 받기 실패: {got.stderr.strip()[-300:]}")
    return got.returncode == 0


def stage_check(ctx: Ctx) -> dict:
    code = "import torch;print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())"
    out = subprocess.run([*ctx.python(), "-c", code], capture_output=True, text=True, env=ctx.env())
    log(f"torch: {out.stdout.strip() or out.stderr.strip()[-300:]}")
    if "True" not in out.stdout:
        raise RuntimeError("GPU가 안 보임. CUDA_VISIBLE_DEVICES와 torch 설치를 확인")
    usable, skipped = [], {}
    for model, needs in NEEDS.items():
        statements = [f"import {m}" for m in needs]
        tried: set[str] = set()
        problem = first_problem(ctx, statements)
        while problem and len(tried) < MAX_FETCH:
            top = problem[1].split(".")[0]
            if not top or top in HEAVY or top in tried:
                break
            tried.add(top)
            package = needs.get(top) or top
            log(f"{model}: {top} 없음 → {package}를 {ctx.work / 'pylib'}에 받음")
            if not fetch(ctx, package):
                break
            problem = first_problem(ctx, statements)
        if problem:
            skipped[model] = [problem[1] or problem[0]]
            log(f"{model}: '{problem[0]}'가 안 돼 이 모델 학습은 뺌 → {problem[2]}")
        else:
            usable.append(model)
    if not usable:
        raise RuntimeError(f"돌릴 수 있는 모델이 없음: {skipped}")
    return {"models": usable, "skipped": skipped, "torch": out.stdout.strip()}


def md5_of(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def stage_data(ctx: Ctx) -> dict:
    expect = json.loads((HERE / "md5.json").read_text(encoding="utf-8"))
    problems = []
    for name, info in expect.items():
        path = ctx.args.upload / name
        if not path.is_file():
            problems.append(f"{name}: 없음")
        elif path.stat().st_size != info["bytes"]:
            problems.append(
                f"{name}: 크기 {path.stat().st_size} (맞는 값 {info['bytes']}). 덜 올라감"
            )
        elif md5_of(path) != info["md5"]:
            problems.append(f"{name}: md5가 다름. 다시 올려야 함")
        else:
            log(f"{name}: 확인")
    if problems:
        raise RuntimeError("올린 파일 문제 → " + " / ".join(problems))
    for name in expect:
        with zipfile.ZipFile(ctx.args.upload / name) as zf:
            zf.extractall(ctx.work)
        log(f"{name}: 풂")
    songs = {}
    for name in ("train457", *EVAL_SETS):
        songs[name] = len(
            json.loads((ctx.data / f"{name}.json").read_text(encoding="utf-8"))["songs"]
        )
    return {"songs": songs}


def stage_egmd(ctx: Ctx) -> dict:
    """E-GMD 어쿠스틱 킷 학습 분할 목록. 운영 코드의 로더(ssaplash_ai)로 정답을 읽음."""
    root = ctx.args.egmd_root
    if not (root / "e-gmd-v1.0.0.csv").is_file():
        log(f"E-GMD가 없음: {root}. E-GMD를 섞는 학습(_RE)은 뺌")
        return {"ok": False, "reason": f"{root}에 e-gmd-v1.0.0.csv 없음"}
    text = (HERE / "egmd_train_acoustic.yaml").read_text(encoding="utf-8")
    lines = [f"root: {root}" if ln.startswith("root:") else ln for ln in text.splitlines()]
    yaml_path = ctx.work / "egmd_train_acoustic.yaml"
    yaml_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    cmd = [*ctx.python(), str(HERE / "export_manifest.py"), "--dataset", str(yaml_path),
           "--name", "egmd_train", "--audio", "original", "--out", str(ctx.work / "egmd")]  # fmt: skip
    with open(ctx.work / "logs" / "egmd.log", "a", encoding="utf-8") as fh:
        code = subprocess.run(
            cmd, cwd=ctx.args.ai, env=ctx.env(), stdout=fh, stderr=subprocess.STDOUT
        )
    manifest = ctx.work / "egmd" / "egmd_train.json"
    if code.returncode != 0 or not manifest.is_file():
        log("E-GMD 목록 만들기 실패(logs/egmd.log). E-GMD를 섞는 학습(_RE)은 뺌")
        return {"ok": False, "reason": "export_manifest 실패"}
    n = len(json.loads(manifest.read_text(encoding="utf-8"))["songs"])
    log(f"E-GMD 목록 {n}곡")
    if n < 100:
        log("E-GMD 곡이 100곡보다 적음. E-GMD를 섞는 학습(_RE)은 뺌")
        return {"ok": False, "reason": f"곡 {n}개뿐", "songs": n}
    return {"ok": True, "songs": n, "manifest": str(manifest)}


def read_done(ctx: Ctx, stage: str) -> dict:
    path = ctx.done / f"{stage}.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}


def active_runs(ctx: Ctx) -> dict[str, dict]:
    check, egmd = (read_done(ctx, s) for s in ("check", "egmd"))
    wanted = tuple(x for x in ctx.args.runs.split(",") if x)  # 비우면 전부
    out = {}
    for name, spec in run_specs(ctx.args).items():
        if spec["model"] not in check["models"] or (spec["egmd"] and not egmd.get("ok")):
            continue
        if wanted and name not in wanted:
            continue
        out[name] = spec
    return out


def train_cmd(ctx: Ctx, name: str, spec: dict) -> list[str]:
    # 구간을 뽑는 비율: E-GMD(share), 나머지는 실제 곡
    share = spec["share"] if spec["egmd"] else 0.0
    real = ctx.data / "train457.json"
    cmd = [*ctx.python(), str(HERE / spec["script"]), "train", *spec["train"],
           "--train", f"{real}:{1 - share:g}" if share > 0 else str(real)]  # fmt: skip
    if share > 0:
        cmd += ["--train", f"{ctx.work / 'egmd' / 'egmd_train.json'}:{share:g}"]
    return [*cmd, "--val", str(ctx.data / "val10.json"), "--out", str(ctx.runs / name)]


def dump_cmd(ctx: Ctx, name: str, spec: dict) -> list[str]:
    best = str(ctx.runs / name / "best.pt")
    sets = [x for n in EVAL_SETS for x in ("--manifest", str(ctx.data / f"{n}.json"))]
    cmd = [*ctx.python(), str(HERE / spec["script"]), "dump", *spec["dump"], "--checkpoint", best]
    return [*cmd, "--tag", name, "--acts", str(ctx.work / "acts"), *sets]


def run_parallel(ctx: Ctx, kind: str, cmds: dict[str, list[str]]) -> dict:
    """명령을 전부 동시에 실행하고 끝날 때까지 기다림. 이미 끝난 것은 건너뜀. 실패한 것은 기록만 하고 계속."""
    procs, result, waiting = {}, {}, []
    for name, cmd in cmds.items():
        if (ctx.done / f"{kind}_{name}.json").is_file():
            result[name] = "이미 끝남"
        else:
            waiting.append((name, cmd))
    limit = ctx.args.parallel or len(waiting)
    while procs or waiting:
        while waiting and len(procs) < limit:
            name, cmd = waiting.pop(0)
            fh = open(ctx.work / "logs" / f"{kind}_{name}.log", "a", encoding="utf-8")
            proc = subprocess.Popen(
                cmd, cwd=ctx.work, env=ctx.env(), stdout=fh, stderr=subprocess.STDOUT
            )
            procs[name] = (proc, fh)
            log(f"{kind} {name}: 시작 (pid {proc.pid})")
        time.sleep(ctx.args.poll)
        for name in list(procs):
            proc, fh = procs[name]
            code = proc.poll()
            if code is None:
                continue
            fh.close()
            del procs[name]
            finished = kind == "train" and (ctx.runs / name / "done.json").is_file()
            if code == 0 or finished:
                (ctx.done / f"{kind}_{name}.json").write_text(
                    json.dumps({"ok": True}), encoding="utf-8"
                )
                result[name] = "끝"
            else:
                result[name] = f"실패 (exit {code})"
            log(f"{kind} {name}: {result[name]}")
    return result


def stage_train(ctx: Ctx) -> dict:
    runs = active_runs(ctx)
    ctx.save_state(runs=list(runs))
    result = run_parallel(ctx, "train", {n: train_cmd(ctx, n, s) for n, s in runs.items()})
    if not any(v in ("끝", "이미 끝남") for v in result.values()):
        raise RuntimeError(f"끝난 학습이 없음: {result}")
    return {"runs": result}


def stage_dump(ctx: Ctx) -> dict:
    runs = {n: s for n, s in active_runs(ctx).items() if (ctx.runs / n / "best.pt").is_file()}
    return {"runs": run_parallel(ctx, "dump", {n: dump_cmd(ctx, n, s) for n, s in runs.items()})}


def stage_pack(ctx: Ctx) -> dict:
    files = [ctx.state_path, ctx.work / "pipeline.log"]
    files += sorted(ctx.done.glob("*.json")) + sorted((ctx.work / "logs").glob("*.log"))
    for run in sorted(ctx.runs.iterdir()):
        files += [run / "train_log.jsonl", run / "args.json"]
    if (ctx.work / "acts").is_dir():
        files += sorted(p for p in (ctx.work / "acts").rglob("*.npz"))
    files = [p for p in files if p.is_file()]
    ctx.args.results.mkdir(parents=True, exist_ok=True)
    stem = ctx.args.zip_name
    dest = ctx.args.results / f"{stem}.zip"
    tmp = dest.with_suffix(".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in files:
            zf.write(p, f"{stem}/{p.relative_to(ctx.work).as_posix()}")
    tmp.replace(dest)
    log(f"묶음 {dest} ({dest.stat().st_size / 2**20:.1f}MB, 파일 {len(files)}개)")
    return {"zip": str(dest), "files": len(files)}


STAGES = {
    "check": stage_check,
    "data": stage_data,
    "egmd": stage_egmd,
    "train": stage_train,
    "dump": stage_dump,
    "pack": stage_pack,
}


def cmd_run(args: argparse.Namespace) -> int:
    ctx = Ctx(args)
    ctx.save_state(pid=os.getpid(), started=time.strftime("%Y-%m-%d %H:%M:%S"), current="시작")
    for stage, fn in STAGES.items():
        if ctx.is_done(stage):
            info = json.loads((ctx.done / f"{stage}.json").read_text(encoding="utf-8"))
            # 패키지가 없어 빠진 모델이 있으면, 학습을 시작하기 전까지는 다시 실행할 때 패키지를 다시 확인함
            if stage == "check" and info.get("skipped") and not ctx.is_done("train"):
                log("check: 빠진 모델이 있어 다시 확인")
            else:
                log(f"{stage}: 이미 끝남, 건너뜀")
                continue
        t0 = time.time()
        log(f"{stage}: 시작")
        ctx.save_state(current=stage, current_started=time.strftime("%Y-%m-%d %H:%M:%S"))
        info = fn(ctx)
        info["minutes"] = round((time.time() - t0) / 60, 1)
        ctx.mark_done(stage, info)
    ctx.save_state(current="끝", finished=time.strftime("%Y-%m-%d %H:%M:%S"))
    log("전체 끝")
    return 0


# ---------- 진행 확인 ----------


def pid_alive(pid) -> bool:
    if not pid:
        return False
    try:
        os.kill(int(pid), 0)
    except (OSError, ValueError):
        return False
    # 죽었는데 노트북 커널이 아직 거두지 않은 프로세스(상태 Z)는 죽은 것으로 봄
    try:
        stat = Path(f"/proc/{int(pid)}/stat").read_text()
    except OSError:
        return True
    return stat.rsplit(")", 1)[-1].split()[:1] != ["Z"]


def tail(path: Path, n: int = 1) -> list[str]:
    if not path.is_file():
        return []
    with open(path, "rb") as fh:
        fh.seek(0, 2)
        fh.seek(max(fh.tell() - 4000, 0))
        text = fh.read().decode("utf-8", errors="replace")
    return [ln for ln in text.splitlines() if ln.strip()][-n:]


def cmd_status(args: argparse.Namespace) -> int:
    work = args.work
    sp = work / "state.json"
    if not sp.is_file():
        print("아직 시작 안 함")
        return 0
    st = json.loads(sp.read_text(encoding="utf-8"))
    alive = pid_alive(st.get("pid"))
    word = "도는 중" if alive else "끝남" if st.get("current") == "끝" else "멈춤"
    print(f"파이프라인 {word} (pid {st.get('pid')}, 시작 {st.get('started')})")
    print(f"지금 단계: {st.get('current')} (시작 {st.get('current_started')})")
    for stage in STAGES:
        d = work / "done" / f"{stage}.json"
        if d.is_file():
            info = json.loads(d.read_text(encoding="utf-8"))
            brief = {
                k: v
                for k, v in info.items()
                if k in ("minutes", "models", "skipped", "songs", "ok", "reason", "runs", "zip")
            }
            print(f"  [끝] {stage} {brief}")
        else:
            print(f"  [ ] {stage}")
    for name in st.get("runs", []):
        logf = work / "runs" / name / "train_log.jsonl"
        rows = (
            [json.loads(x) for x in logf.read_text(encoding="utf-8").splitlines()]
            if logf.is_file()
            else []
        )
        line = (tail(work / "logs" / f"train_{name}.log") or [""])[0][:110]
        if rows:
            best = max(rows, key=lambda r: r["val"]["f1"])
            print(
                f"  {name}: 학습 전 {rows[0]['val']['f1']:.3f} → 최고 {best['val']['f1']:.3f} "
                f"(스텝 {best['step']}), 마지막 검증 스텝 {rows[-1]['step']} | {line}"
            )
        else:
            print(f"  {name}: 검증 기록 아직 없음 | {line}")
    if st.get("current") != "끝":
        print(f"지금 시각 {time.strftime('%Y-%m-%d %H:%M:%S')}. 파이프라인 로그 끝:")
        for ln in tail(work / "pipeline.log", 4):
            print("   ", ln[:300])
    return 0


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    sub = ap.add_subparsers(dest="cmd", required=True)
    r = sub.add_parser("run")
    r.add_argument("--work", type=Path, required=True)
    r.add_argument("--upload", type=Path, required=True, help="올린 zip이 있는 폴더")
    r.add_argument(
        "--ai", type=Path, required=True, help="운영 코드 AI 폴더(ssaplash_ai를 불러올 곳)"
    )
    r.add_argument("--vpy", required=True, help="실행할 python (운영 .venv)")
    r.add_argument("--uv", default="uv", help="uv 실행 명령. 예: 'python -m uv'")
    r.add_argument("--egmd-root", type=Path, required=True)
    r.add_argument("--results", type=Path, required=True)
    r.add_argument("--runs", default="", help="쉼표로 구분한 학습 이름만 실행. 비우면 전부")
    r.add_argument("--zip-name", default="ext_ft_results", help="결과 zip 이름(확장자 제외)")
    r.add_argument("--scale", type=float, default=1.0, help="스텝 수 배수(동작 확인용)")
    r.add_argument("--parallel", type=int, default=0, help="동시에 돌릴 학습 수. 0이면 전부")
    r.add_argument("--poll", type=float, default=15.0, help="학습이 끝났는지 보는 간격(초)")
    s = sub.add_parser("status")
    s.add_argument("--work", type=Path, required=True)
    args = ap.parse_args()
    for key in ("work", "upload", "ai", "egmd_root", "results"):
        if getattr(args, key, None) is not None:
            setattr(args, key, Path(getattr(args, key)).expanduser().resolve())
    return cmd_run(args) if args.cmd == "run" else cmd_status(args)


if __name__ == "__main__":
    raise SystemExit(main())''',
    "egmd_train_acoustic.yaml": r'''
# E-GMD train 분할 중 어쿠스틱 음색 킷 26개 (egmd_test_acoustic.yaml과 같은 킷 목록). beat + fill 21,294개, 206시간.
# 채보 모델 탐 파인튜닝의 학습 데이터. test 분할(평가용)과 연주가 겹치지 않음.
# 전체 zip(96GB)을 data/e-gmd/ 에 풀어 e-gmd-v1.0.0/ 안에 csv와 drummer 폴더가 오게 둔다. 서버에만 있음.
name: egmd_train_acoustic
kind: egmd
root: data/e-gmd/e-gmd-v1.0.0
subset: train
expected_songs: 21294
sample_rate: 44100
clip: null
songs: null
extra:
  beat_type: all
  kits:
  - Acoustic Kit
  - Studio (Live Room)
  - Classic Rock
  - Jazz Funk
  - ClassicMetal (80s-90s)
  - 60s Rock
  - Modern Funk
  - Fat Rock (Power Toms)
  - Pop-Rock (Studio)
  - Dry & Heavy (Folk Rock)
  - Second Line
  - Heavy Metal
  - Arena Stage
  - Warmer Funk
  - Jazz
  - Live Rock
  - Shuffle (Blues)
  - Alternative (METAL)
  - West Coast (FUNK)
  - Live Fusion
  - Speed Metal
  - Bigga Bop (Jazz)
  - Funk Rock
  - Alternative (Rock)
  - Tight Prog
  - Unplugged''',
    "md5.json": r'''
{
 "ext_ft_data_01.zip": {
  "md5": "b1cb05b3dfff3c219fb1e4197e938926",
  "bytes": 938420857
 },
 "ext_ft_data_02.zip": {
  "md5": "d30687ec3b68a4c045f19591f3c827d0",
  "bytes": 940614202
 },
 "ext_ft_data_03.zip": {
  "md5": "f1abe044d11daeb1c9f03de8ebbd147d",
  "bytes": 940242493
 },
 "ext_ft_data_04.zip": {
  "md5": "5649e5e2660dde636ed678774bac56b5",
  "bytes": 939514562
 },
 "ext_ft_data_05.zip": {
  "md5": "ffb356376319a73d3e87675bd0cd3272",
  "bytes": 937157635
 },
 "ext_ft_data_06.zip": {
  "md5": "1a8059178b5604ecbf4e7ff9a836ba83",
  "bytes": 929622109
 },
 "ext_ft_data_07.zip": {
  "md5": "d98966c849dd41c7290016accfa03f79",
  "bytes": 929678673
 },
 "ext_ft_data_08.zip": {
  "md5": "e74c7f2a637203cb6d1c8a000cee3617",
  "bytes": 620982057
 },
 "ext_ft_models.zip": {
  "md5": "12458df018dac1dfd720cd2f20431c97",
  "bytes": 629275993
 }
}''',
}


def running_pid():
    """돌고 있는 파이프라인 pid. 없으면 None."""
    pid_file = EXT / "launcher.pid"
    if not pid_file.is_file():
        return None
    pid = pid_file.read_text().strip()
    try:
        cmdline = Path(f"/proc/{pid}/cmdline").read_bytes()
    except OSError:
        return None
    return pid if b"ext_ft_pipeline" in cmdline else None


pid = running_pid()
if pid:
    print("이미 도는 중. pid", pid, "→ 2번 셀로 진행을 확인함")
else:
    for name, text in FILES_TEXT.items():
        path = FILES / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text.lstrip("\n"), encoding="utf-8")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU_ID, PYTHONUTF8="1", PYTHONUNBUFFERED="1")
    env.pop("PYTHONPATH", None)
    cmd = [
        str(VPY), str(FILES / "ext_ft_pipeline.py"), "run",
        "--work", str(EXT), "--upload", str(WORK / "upload"), "--ai", str(AI), "--vpy", str(VPY),
        "--egmd-root", str(WORK / "datasets" / "e-gmd" / "e-gmd-v1.0.0"),
        "--results", str(WORK / "results"), "--runs", RUNS, "--zip-name", "ext_ft_results",
        "--uv", f"{sys.executable} -m uv",       # 없는 패키지를 받을 때만 씀
    ]
    proc = subprocess.Popen(["nohup", "nice", "-n", "10", *cmd], cwd=EXT, env=env,
                            stdout=open(EXT / "pipeline.log", "a"), stderr=subprocess.STDOUT,
                            start_new_session=True)
    (EXT / "launcher.pid").write_text(str(proc.pid))
    print("시작. pid", proc.pid, "로그", EXT / "pipeline.log")
# 확인: 처음엔 '시작. pid 숫자 로그 /home/.../ssaplash/ext_ft/pipeline.log'.
#       다시 치면 '이미 도는 중. pid 숫자'. 1분쯤 뒤 2번 셀에서 check가 [끝]이 되는지 봄


In [ ]:
# 2. 진행 확인 (여러 번 쳐도 됨. 커널이 다시 시작돼도 이 셀만 바로 쳐도 됨)
from pathlib import Path

EXT = Path.home() / "ssaplash" / "ext_ft"
VPY = Path.home() / "ssaplash" / "S15P31A307" / "AI" / ".venv" / "bin" / "python"
!{VPY} {EXT}/files/ext_ft_pipeline.py status --work {EXT}
!nvidia-smi -i 6 --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader
!ls -la ~/ssaplash/results/ext_ft_results.zip 2>/dev/null || echo "결과 zip 아직 없음"
# 확인 (학습 중 예):
#   파이프라인 도는 중 (pid 12345, 시작 2026-11-11 15:00:00)
#   지금 단계: train (시작 2026-11-11 15:20:00)
#     [끝] check {'models': ['strum'], 'skipped': {}, 'minutes': 0.2}
#     [끝] data {'songs': {'train457': 457, 'val10': 10, ...}, 'minutes': 6.0}
#     [끝] egmd {'ok': True, 'songs': 2만 안팎, 'minutes': 10.0}
#     [ ] train   [ ] dump   [ ] pack
#     strum_p3_R: 학습 전 0.739 → 최고 0.85x (스텝 1500), 마지막 검증 스텝 2000 | [train] 스텝 2100/8000 손실 ...
#     (학습 2개가 한 줄씩)
# - 단계가 모두 [끝]이고 zip 크기가 보이면 끝. ~/ssaplash/results/ext_ft_results.zip을 내려받음
# - '멈춤'이 보이면 아래 '파이프라인 로그 끝' 줄을 적어 두고 1번 셀을 다시 실행. 같은 오류로 또 멈추면 그 줄을 알려 줌
# - egmd가 {'ok': False}면 strum_p3_RE는 빠지고 strum_p3_R만 돎
